# Thesis — ESCP Phase 1 & 2 on Google Colab

This notebook clones the repo from GitHub, installs Gurobi, activates a **WLS academic license**, and runs the Phase-1 batch.

**Workflow:** edit code locally → `git push` → re-run the *Get the code* cell here (`git pull`).

**License:** you need a free WLS academic license from https://portal.gurobi.com (Named-user academic licenses do NOT work in the cloud).

## 1. Install Gurobi

In [ ]:
!pip install -q gurobipy

## 2. Activate the WLS license

Get these three values from your WLS license on the Gurobi portal.

**Tip:** to avoid pasting secrets into the notebook, store them with Colab *Secrets* (the 🔑 icon in the left sidebar) named `WLSACCESSID`, `WLSSECRET`, `LICENSEID`, then use the commented `userdata` block.

In [ ]:
# Option A: paste directly (quick, but secrets live in the notebook)
WLSACCESSID = "PASTE_YOUR_WLSACCESSID"
WLSSECRET   = "PASTE_YOUR_WLSSECRET"
LICENSEID   = "0"  # e.g. "1234567"

# Option B (recommended): read from Colab Secrets
# from google.colab import userdata
# WLSACCESSID = userdata.get('WLSACCESSID')
# WLSSECRET   = userdata.get('WLSSECRET')
# LICENSEID   = userdata.get('LICENSEID')

# Write a gurobi.lic file so the DEFAULT Gurobi env (used by build_model)
# picks up the WLS credentials automatically — no code changes needed.
import os
lic_path = '/content/gurobi.lic'
with open(lic_path, 'w') as f:
    f.write(f'WLSACCESSID={WLSACCESSID}\n')
    f.write(f'WLSSECRET={WLSSECRET}\n')
    f.write(f'LICENSEID={LICENSEID}\n')
os.environ['GRB_LICENSE_FILE'] = lic_path

# Quick license check (uses the default env)
import gurobipy as gp
_m = gp.Model()
print('Gurobi license OK')

## 3. Get the code (clone or pull)

Re-run this cell whenever you push new commits locally.

In [ ]:
import os, sys

REPO_URL = 'https://github.com/ekaterina1tum/Bachelor-Thesis.git'
REPO_DIR = '/content/Bachelor-Thesis'

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

src = os.path.join(REPO_DIR, 'src')
if src not in sys.path:
    sys.path.insert(0, src)

print('src on path:', src)
print(os.listdir(src))

## 4. Smoke test: solve one instance

In [ ]:
from instance import load_instance
from graph import build_graph
from model import build_model

DATA = os.path.join(REPO_DIR, 'data', 'MSCDPinstances', '025')

inst = load_instance(os.path.join(DATA, '025_C101.txt'), max_shift=480.0)
g = build_graph(inst)
mdl = build_model(inst, g)
mdl.setParam('TimeLimit', 120)
mdl.optimize()
print('Objective:', mdl.ObjVal)

## 5. Run the Phase-1 batch (one type at a time)

Args: `<folder> <time_limit_s> <max_shift> <type_filter>`. Type filter is one of `C1 C2 R1 R2 RC1 RC2`. Omit the filter to run all 56 instances.

In [ ]:
!cd {src} && python phase1_batch.py {DATA} 3600 480 C1